<a href="https://colab.research.google.com/github/fernandaelenn/fernandaelenn/blob/main/N%C3%ADvel_RIO_ANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAVIO-VECTOR — extração de níveis do rio (ANA / HidroWeb)
Tutorial para extração de dados de Nível de Rio referente a Base da Agencia Nacional de Água:

Use o arquivo **CODIGO_Cotas.csv**, extraído do ZIP do HidroWeb.

O notebook lê um arquivo de uma estação por vez. Você informa o caminho no Drive e as datas de coleta.
Ele usa a média diária já publicada pela ANA (`MediaDiaria = 1`) e extrai o dia da coleta e exatamente 15, 30, 45 e 60 dias antes.

**Não há soma/acúmulo nem média de uma janela de 15/30/45/60 dias.** Cota é o nível em relação ao zero da régua da estação.
No formato de CSV utilizado aqui, as cotas estão em centímetros e são convertidas para metros.
O período manhã/tarde/noite da coleta não altera a seleção da média diária.

Quando há versões bruta e consistida para a mesma data, a consistida tem prioridade; elas nunca são somadas ou promediadas.
Dados ausentes permanecem vazios; valores negativos e zero são preservados.
Este notebook não escolhe automaticamente a estação a partir das coordenadas: confira a estação de referência antes de usar o arquivo.
Fonte: https://www.snirh.gov.br/hidroweb/serieshistoricas


## 1. Carregar os pacotes

In [2]:
from pathlib import Path
from datetime import datetime
import csv
import io
import re
import pandas as pd

## 2. Conectar ao Google Drive
Autorize o acesso quando o Colab solicitar.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Informar o caminho do CSV
Cole o **caminho do arquivo**, não o link de compartilhamento do Drive.
Exemplo: `/content/drive/MyDrive/0 MESTRADO/DADOS BRUTOS/66825000_Cotas.csv`
No painel de arquivos do Colab, encontre o CSV e use **Copiar caminho**.


In [6]:
# Caminho do CSV no Google Drive
CAMINHO_CSV = Path(
    "/content/drive/MyDrive/0 MESTRADO/DADOS BRUTOS/NIVEL DE RIO/66070004_Cotas_Caceres.csv"
)

print("Arquivo selecionado:", CAMINHO_CSV)

Arquivo selecionado: /content/drive/MyDrive/0 MESTRADO/DADOS BRUTOS/NIVEL DE RIO/66070004_Cotas_Caceres.csv


## 4. Ler e organizar as cotas diárias
O CSV da ANA contém um preâmbulo e linhas mensais, com `Cota01` a `Cota31`.
Essas colunas são transformadas em datas individuais. `CotaXXStatus` é a condição do registro, não o nível.

Status: 0 = branco; 1 = real; 2 = estimado; 3 = duvidoso; 4 = régua seca.
Status vazio não é interpretado como confirmação de dado real. Régua seca é mantida como nível ausente.
Se não houver média diária publicada, o resultado fica ausente; o notebook não calcula uma média alternativa silenciosamente.


In [7]:
def ler_csv_ana(caminho):
    raw = Path(caminho).read_bytes()
    try:
        texto = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        texto = raw.decode("cp1252")
    linhas = texto.splitlines()
    inicio = next((i for i, linha in enumerate(linhas)
                   if linha.lstrip().lstrip('"').startswith("EstacaoCodigo")), None)
    if inicio is None:
        raise ValueError("Cabeçalho EstacaoCodigo não encontrado. Use o CSV original de Cotas da ANA.")
    separador = ";" if ";" in linhas[inicio] else ","
    tabela = pd.read_csv(io.StringIO("\n".join(linhas[inicio:])), sep=separador,
                        dtype=str, keep_default_na=False)
    tabela.columns = tabela.columns.str.strip()
    obrigatorias = {"EstacaoCodigo", "NivelConsistencia", "Data", "MediaDiaria"}
    obrigatorias.update(f"Cota{dia:02d}" for dia in range(1, 32))
    faltantes = obrigatorias - set(tabela.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes: {sorted(faltantes)}. Confira se o arquivo é de Cotas.")
    for coluna in tabela.columns:
        tabela[coluna] = tabela[coluna].str.strip()
    return tabela

def numero_ana(valor):
    texto = str(valor).strip()
    if texto.lower() in {"", "nan", "null", "none"}:
        return float("nan")
    return float(texto.replace(",", "."))

def organizar_medias_diarias(tabela, estacao):
    registros = []
    selecionada = tabela[tabela["EstacaoCodigo"] == str(estacao)]
    for _, linha in selecionada.iterrows():
        if numero_ana(linha["MediaDiaria"]) != 1:
            continue
        mes = pd.to_datetime(linha["Data"], format="%d/%m/%Y", errors="raise")
        if mes.day != 1:
            raise ValueError(f"Data mensal inesperada: {linha['Data']}. Confira o formato do arquivo.")
        consistencia = int(numero_ana(linha["NivelConsistencia"]))
        if consistencia not in (1, 2):
            raise ValueError(f"Nível de consistência desconhecido: {consistencia}")
        for dia in range(1, mes.days_in_month + 1):
            coluna = f"Cota{dia:02d}"
            status = str(linha.get(coluna + "Status", "")).strip()
            if status:
                status = str(int(numero_ana(status)))
                if status not in {"0", "1", "2", "3", "4"}:
                    raise ValueError(f"Status desconhecido: {status}")
            cota_cm = numero_ana(linha[coluna])
            if status == "4":
                cota_cm = float("nan")
            registros.append({"data": mes.replace(day=dia), "cota_m": cota_cm / 100,
                              "nivel_consistencia": consistencia, "status_ana": status})
    if not registros:
        raise ValueError("Esta estação não tem linhas MediaDiaria = 1 no arquivo. Não foi calculada média alternativa.")
    diarios = pd.DataFrame(registros)
    prioridade = diarios.groupby("data")["nivel_consistencia"].transform("max")
    candidatos = diarios[diarios["nivel_consistencia"] == prioridade]
    distintos = candidatos.drop_duplicates(["data", "cota_m", "status_ana"])
    conflitos = distintos[distintos.duplicated("data", keep=False)]
    if not conflitos.empty:
        data = conflitos.iloc[0]["data"]
        raise ValueError(f"Registros diários conflitantes em {data:%d/%m/%Y}. Confira as linhas na mesma versão.")
    return distintos.drop_duplicates("data").set_index("data").sort_index()

def extrair_defasagens(diarios, datas, estacao):
    campos = {0: "river_level_day", 15: "river_level_15_days_before",
              30: "river_level_30_days_before", 45: "river_level_45_days_before",
              60: "river_level_60_days_before"}
    resultado, detalhes = [], []
    for data in sorted(set(pd.Timestamp(d).normalize() for d in datas)):
        saida = {"date": data.strftime("%Y-%m-%d"), "river_station": str(estacao), "unit": "m"}
        for defasagem, campo in campos.items():
            referencia = data - pd.Timedelta(days=defasagem)
            nivel, consistencia, status = float("nan"), "", ""
            if referencia in diarios.index:
                linha = diarios.loc[referencia]
                nivel = linha["cota_m"]
                consistencia = int(linha["nivel_consistencia"])
                status = linha["status_ana"]
            saida[campo] = nivel
            detalhes.append({"date": saida["date"], "river_station": str(estacao),
                             "lag_days": defasagem, "reference_date": referencia.strftime("%Y-%m-%d"),
                             "river_level_m": nivel, "nivel_consistencia": consistencia,
                             "status_ana": status, "available": bool(pd.notna(nivel))})
        resultado.append(saida)
    return pd.DataFrame(resultado), pd.DataFrame(detalhes)

tabela_ana = ler_csv_ana(CAMINHO_CSV)
estacoes = sorted(set(tabela_ana["EstacaoCodigo"]) - {""})
print("Códigos de estação no arquivo:", ", ".join(estacoes))
if len(estacoes) == 1:
    ESTACAO = estacoes[0]
else:
    ESTACAO = input("Digite o código ANA da estação que deseja usar: ").strip()
    if ESTACAO not in estacoes:
        raise ValueError("Código não encontrado no arquivo.")
NOME_ESTACAO = input("Nome da estação (opcional, para documentar a origem): ").strip()
medias_diarias = organizar_medias_diarias(tabela_ana, ESTACAO)
print(f"Estação selecionada: {ESTACAO} {NOME_ESTACAO}")
print(f"Meses representados: {medias_diarias.index.min():%d/%m/%Y} a {medias_diarias.index.max():%d/%m/%Y}")
print("Dias com valores:", medias_diarias["cota_m"].notna().sum())
print("Critério: média diária publicada; consistido tem prioridade sobre bruto; unidade de saída: metros.")
display(medias_diarias.tail(5))


Códigos de estação no arquivo: 66070004
Nome da estação (opcional, para documentar a origem): Caceres
Estação selecionada: 66070004 Caceres
Meses representados: 01/12/1965 a 31/03/2026
Dias com valores: 22024
Critério: média diária publicada; consistido tem prioridade sobre bruto; unidade de saída: metros.


,cota_m,nivel_consistencia,status_ana
data,,,
2026-03-27,4.46,1,
2026-03-28,4.48,1,
2026-03-29,4.48,1,
2026-03-30,4.45,1,
2026-03-31,4.44,1,


## 5. Selecionar as datas e extrair os resultados
Digite datas no formato **DD/MM/AAAA**, separadas por vírgula ou ponto e vírgula.
Exemplo: `21/03/2026, 22/03/2026`.
Use a data da coleta; as defasagens serão calculadas automaticamente.


In [8]:
entrada_datas = input("Datas de coleta (DD/MM/AAAA): ").strip()
textos_datas = [item.strip() for item in re.split(r"[,;]", entrada_datas) if item.strip()]
if not textos_datas:
    raise ValueError("Informe pelo menos uma data.")
DATAS_COLETA = [datetime.strptime(item, "%d/%m/%Y") for item in textos_datas]
resultado, auditoria = extrair_defasagens(medias_diarias, DATAS_COLETA, ESTACAO)
resultado.insert(2, "river_station_name", NOME_ESTACAO)
print("Níveis em metros — um registro por data de coleta:")
display(resultado.style.format({col: "{:.2f}" for col in resultado.columns if col.startswith("river_level")}, na_rep="Ausente"))
print("Datas de referência, consistência e status de cada valor:")
display(auditoria)
ausentes = auditoria[~auditoria["available"]]
if not ausentes.empty:
    print("ATENÇÃO: há datas sem nível disponível. Serão salvas como células vazias, sem substituição por zero.")
    display(ausentes)
if auditoria["status_ana"].isin(["2", "3", "4"]).any():
    print("ATENÇÃO: há status estimado, duvidoso ou régua seca. Confira a tabela de auditoria.")
if (auditoria["status_ana"] == "").any():
    print("Há status vazios; eles não foram interpretados como 'Real'.")


Datas de coleta (DD/MM/AAAA): 21/03/2026
Níveis em metros — um registro por data de coleta:


,date,river_station,river_station_name,unit,river_level_day,river_level_15_days_before,river_level_30_days_before,river_level_45_days_before,river_level_60_days_before
0,2026-03-21,66070004,Caceres,m,4.45,4.07,3.22,2.72,2.03


Datas de referência, consistência e status de cada valor:


,date,river_station,lag_days,reference_date,river_level_m,nivel_consistencia,status_ana,available
0,2026-03-21,66070004,0,2026-03-21,4.45,1,,True
1,2026-03-21,66070004,15,2026-03-06,4.07,1,,True
2,2026-03-21,66070004,30,2026-02-19,3.22,1,,True
3,2026-03-21,66070004,45,2026-02-04,2.72,1,,True
4,2026-03-21,66070004,60,2026-01-20,2.03,1,,True


Há status vazios; eles não foram interpretados como 'Real'.


## 6. Salvar no Drive e baixar
O primeiro arquivo tem as cinco colunas para o dashboard. O segundo documenta a data de referência, a consistência e o status de cada valor.
Ambos usam separador `;` e **ponto decimal**. Os nomes têm data e hora para evitar substituir resultados anteriores.


In [ ]:
pasta_padrao = CAMINHO_CSV.parent / "ANA_RESULTADOS"
entrada_pasta = input(f"Pasta de saída (Enter para {pasta_padrao}): ").strip().strip('"').strip("'")
PASTA_SAIDA = Path(entrada_pasta) if entrada_pasta else pasta_padrao
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
identificador = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
caminho_resultado = PASTA_SAIDA / f"niveis_rio_{ESTACAO}_{identificador}.csv"
caminho_auditoria = PASTA_SAIDA / f"referencias_niveis_rio_{ESTACAO}_{identificador}.csv"
resultado.to_csv(caminho_resultado, sep=";", decimal=".", index=False,
                 encoding="utf-8-sig", float_format="%.2f", na_rep="")
auditoria.to_csv(caminho_auditoria, sep=";", decimal=".", index=False,
                 encoding="utf-8-sig", float_format="%.2f", na_rep="")
print("Resultado salvo:", caminho_resultado)
print("Referências salvas:", caminho_auditoria)
if input("Baixar o CSV de resultados para o computador? (s/n): ").strip().lower() in {"s", "sim"}:
    from google.colab import files
    files.download(str(caminho_resultado))
